In [ ]:
import pandas as pd
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import chardet

# Définir le chemin du dossier contenant tous les fichiers
data_path = "/home/kaoutar/Desktop/data"

# Charger la table périmètre qui servira de base
perimetre_path = os.path.join(data_path, "Périmétre_projet.txt")
df_perimetre = pd.read_csv(perimetre_path, encoding='latin1', sep='|')

# Initialiser le dataframe final avec la table périmètre
df_final = df_perimetre.copy()
print(f"Table périmètre chargée avec {len(df_perimetre)} lignes et {df_perimetre.shape[1]} colonnes")

# Liste des tables à joindre avec leurs chemins complets
tables_a_joindre = {
    'credit': os.path.join(data_path, "ACT_CD_CREDIT_MDM28032025.txt"),
    'solde': os.path.join(data_path, "ACT_CD_SOLDE_CLIENT_MDM28032025.txt"),
    'operations': os.path.join(data_path, "ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt"),
    'produit': os.path.join(data_path, "ACT_CD_PRODUIT_MDM20032025.txt"),
    'connexion': os.path.join(data_path, "ACT_CD_STAT_CONNEXION_MDM28032025.txt"),
    'transactions': os.path.join(data_path, "ACT_CD_TRANSA_MDM20032025.txt"),
    'equipements': os.path.join(data_path, "ACT_EQUI_MDM28032025.txt"),
    'transferts': os.path.join(data_path, "ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt"),
    'getchq': os.path.join(data_path, "ACT_CD_GETCHQ_MDM28032025.txt"),
    'vignette': os.path.join(data_path, "ACT_CD_PAI_VIGNETTE_MDM20032025.txt")
}

# Fonction pour détecter l'encodage d'un fichier
def detect_encoding(file_path):
    try:
        with open(file_path, 'rb') as f:
            rawdata = f.read(100000)  # Lire les premiers 100KB pour l'échantillonnage
        result = chardet.detect(rawdata)
        encoding = result['encoding']
        confidence = result['confidence']
        print(f"Encodage détecté: {encoding} (confiance: {confidence:.2f})")
        
        if encoding is None or confidence < 0.7:
            encoding = 'latin1'
    except:
        encoding = 'latin1'
    
    return encoding

# Fonction pour détecter le séparateur dans un fichier
def detect_separator(file_path, encoding='latin1'):
    try:
        with open(file_path, 'r', encoding=encoding, errors='replace') as f:
            first_lines = [f.readline() for _ in range(3)]
        
        separators = ['|', ';', ',', '\t']
        counts = {}
        
        for sep in separators:
            counts[sep] = sum(line.count(sep) for line in first_lines)
        
        best_sep = max(counts, key=counts.get)
        if counts[best_sep] > 5:  # Si un séparateur est utilisé fréquemment
            return best_sep
    except:
        pass
    
    return '|'  # Séparateur par défaut

# Fonction pour charger un fichier avec gestion des erreurs
def load_file_safely(file_path, encoding='latin1', separator='|'):
    # Essayer différentes méthodes de chargement
    try:
        # 1. Méthode standard
        return pd.read_csv(file_path, encoding=encoding, sep=separator)
    except Exception as e1:
        print(f"Méthode standard a échoué: {str(e1)}")
        
        try:
            # 2. Méthode avec engine python
            return pd.read_csv(file_path, encoding=encoding, sep=separator, engine='python')
        except Exception as e2:
            print(f"Méthode engine python a échoué: {str(e2)}")
            
            try:
                # 3. Lecture manuelle ligne par ligne
                with open(file_path, 'r', encoding=encoding, errors='replace') as f:
                    lines = f.readlines()
                
                if not lines:
                    raise Exception("Fichier vide")
                
                header = lines[0].strip().split(separator)
                data = []
                
                for line in lines[1:]:
                    fields = line.strip().split(separator)
                    # Ajuster si le nombre de champs ne correspond pas au header
                    if len(fields) > len(header):
                        fields = fields[:len(header)]
                    elif len(fields) < len(header):
                        fields = fields + [''] * (len(header) - len(fields))
                    data.append(fields)
                
                df = pd.DataFrame(data, columns=header)
                print(f"Lecture réussie par méthode manuelle, {len(df)} lignes chargées")
                return df
            except Exception as e3:
                print(f"Échec de toutes les méthodes de lecture: {str(e3)}")
                return pd.DataFrame()  # Retourner un DataFrame vide en cas d'échec

# Traiter chaque table séparément
for table_name, file_path in tables_a_joindre.items():
    print(f"\nTraitement de la table {table_name}...")
    
    try:
        # Déterminer l'encodage pour ce fichier
        encoding = detect_encoding(file_path)
        
        # Déterminer le séparateur à utiliser
        separator = detect_separator(file_path, encoding)
        print(f"Utilisation du séparateur '{separator}' pour la table {table_name}")
        
        # Charger la table avec les paramètres détectés
        df_temp = load_file_safely(file_path, encoding, separator)
        
        if df_temp.empty:
            print(f"Impossible de charger la table {table_name}. Passage à la table suivante.")
            continue
        
        print(f"Table {table_name} chargée avec {len(df_temp)} lignes")
        
        # Nettoyage des noms de colonnes (supprimer espaces/guillemets)
        df_temp.columns = [col.strip().strip('"\'') for col in df_temp.columns]
        
        # Afficher les colonnes pour le débogage
        print(f"Colonnes dans {table_name}: {', '.join(df_temp.columns)}")
        
        # S'assurer que la colonne d'identifiant est présente
        id_column = None
        if 'RADICAL' in df_temp.columns:
            id_column = 'RADICAL'
        elif 'CONTRAT' in df_temp.columns:
            id_column = 'CONTRAT'
            df_temp.rename(columns={'CONTRAT': 'RADICAL'}, inplace=True)
            print(f"Colonne CONTRAT renommée en RADICAL dans {table_name}")
        
        # Si la colonne n'est pas trouvée, essayer des variantes (casse, espaces)
        if id_column is None:
            for col in df_temp.columns:
                if col.upper().strip() in ['RADICAL', 'CONTRAT']:
                    df_temp.rename(columns={col: 'RADICAL'}, inplace=True)
                    id_column = 'RADICAL'
                    print(f"Colonne '{col}' renommée en RADICAL dans {table_name}")
                    break
        
        # Si toujours pas trouvé, vérifier si une colonne contient des identifiants similaires
        if id_column is None:
            # Pour le cas spécifique de operations, qui semble problématique
            if table_name == 'operations':
                # Essayer de récupérer la première colonne comme RADICAL
                first_col = df_temp.columns[0]
                df_temp.rename(columns={first_col: 'RADICAL'}, inplace=True)
                id_column = 'RADICAL'
                print(f"Première colonne '{first_col}' utilisée comme RADICAL dans {table_name}")
            else:
                print(f"ERREUR: Pas de colonne RADICAL ou CONTRAT dans {table_name}!")
                continue
        
        # Pour les tables volumineuses, faire une agrégation simplifiée
        if table_name in ['operations', 'transactions', 'connexion']:
            print(f"Table volumineuse {table_name}, agrégation simplifiée...")
            
            try:
                # Identifier les colonnes numériques
                for col in df_temp.columns:
                    if col != 'RADICAL':
                        try:
                            df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
                        except:
                            pass
                
                # Agrégation des colonnes numériques
                numeric_cols = df_temp.select_dtypes(include=[np.number]).columns.tolist()
                agg_dict = {}
                
                for col in numeric_cols:
                    if col != 'RADICAL':
                        agg_dict[col] = ['sum', 'mean']
                
                if agg_dict:
                    df_agg = df_temp.groupby('RADICAL').agg(agg_dict)
                    # Aplatir les noms de colonnes
                    flat_cols = []
                    for col in df_agg.columns:
                        flat_cols.append(f"{table_name}_{col[0]}_{col[1]}")
                    df_agg.columns = flat_cols
                    df_agg = df_agg.reset_index()
                    df_temp = df_agg
                else:
                    # Si pas de colonnes numériques, juste compter par RADICAL
                    df_count = df_temp.groupby('RADICAL').size().reset_index(name=f'{table_name}_count')
                    df_temp = df_count
            except Exception as e:
                print(f"Erreur lors de l'agrégation: {str(e)}")
                # Fallback: juste prendre les RADICAL uniques
                df_temp = df_temp[['RADICAL']].drop_duplicates()
        else:
            # Pour les tables non volumineuses, préfixer les colonnes (sauf RADICAL)
            cols_to_rename = {col: f"{table_name}_{col}" for col in df_temp.columns if col != 'RADICAL'}
            df_temp = df_temp.rename(columns=cols_to_rename)
        
        # Jointure
        print(f"Jointure avec la table {table_name}...")
        before_join = len(df_final)
        
        # S'assurer que la colonne RADICAL est du même type dans les deux dataframes
        df_final['RADICAL'] = df_final['RADICAL'].astype(str)
        df_temp['RADICAL'] = df_temp['RADICAL'].astype(str)
        
        df_final = df_final.merge(df_temp, on='RADICAL', how='left')
        
        # Vérifier le succès de la jointure
        cols_from_table = [col for col in df_final.columns if col.startswith(f"{table_name}_")]
        
        if cols_from_table:
            missing_after_join = df_final[cols_from_table].isnull().all(axis=1).sum()
            match_count = before_join - missing_after_join
            match_percent = (match_count / before_join) * 100
            
            print(f"Résultat de la jointure avec {table_name}: {match_count} correspondances ({match_percent:.2f}%)")
            
            # Créer un indicateur de présence/absence pour cette table
            df_final[f"has_{table_name}"] = (~df_final[cols_from_table].isnull().all(axis=1)).astype(int)
        else:
            print(f"Attention: Aucune colonne ajoutée après la jointure avec {table_name}")
        
        # Sauvegarde intermédiaire
        if table_name in ['credit', 'solde', 'produit', 'equipements']:
            interim_path = os.path.join(data_path, f"interim_table_after_{table_name}.csv")
            df_final.to_csv(interim_path, index=False)
            print(f"Sauvegarde intermédiaire après {table_name} dans {interim_path}")
        
    except Exception as e:
        print(f"Erreur générale lors du traitement de la table {table_name}: {str(e)}")
        import traceback
        traceback.print_exc()

# Vérifier la taille finale
print(f"\nDimensions de la table finale: {df_final.shape}")

# Sauvegarder la table finale
output_path = os.path.join(data_path, "table_finale_ml.csv")
df_final.to_csv(output_path, index=False)
print(f"Table finale sauvegardée dans {output_path}")

# Afficher un résumé des indicateurs de présence
indicators = [col for col in df_final.columns if col.startswith("has_")]
if indicators:
    presence_summary = df_final[indicators].sum().sort_values(ascending=False)
    print("\nRésumé des correspondances:")
    for table, count in presence_summary.items():
        table_name = table.replace("has_", "")
        percent = (count / len(df_final)) * 100
        print(f"{table_name}: {count} correspondances ({percent:.2f}%)")

Table périmètre chargée avec 1051499 lignes et 8 colonnes

Traitement de la table credit...
Encodage détecté: ascii (confiance: 1.00)
Utilisation du séparateur '|' pour la table credit
Table credit chargée avec 32835 lignes
Colonnes dans credit: RADICAL, NODOSS, CD_MTIF_DOSS, DATE_OCTROI, MONTANT, DUREE, DT_FIN_DOSS_REEL, CAPITAL_RES, TYPE, SOURCE, IMP
Jointure avec la table credit...
Résultat de la jointure avec credit: 31177 correspondances (2.97%)
Sauvegarde intermédiaire après credit dans /home/kaoutar/Desktop/data/interim_table_after_credit.csv

Traitement de la table solde...
Encodage détecté: ascii (confiance: 1.00)
Utilisation du séparateur '|' pour la table solde
Table solde chargée avec 11005299 lignes
Colonnes dans solde: RADICAL, MVTDEB, MVTCRED, NBMVTCRE_MENS, NBMVTDB_MENS, SOLDE, MOIS, ANNEE
Jointure avec la table solde...
Résultat de la jointure avec solde: 1047145 correspondances (99.28%)
Sauvegarde intermédiaire après solde dans /home/kaoutar/Desktop/data/interim_table

In [ ]:
import pandas as pd
import os
import chardet

# Chemin des fichiers
data_path = "/home/kaoutar/Desktop/data"

# Liste des fichiers à traiter
fichiers = {
    'perimetre': ('Périmétre_projet.txt', '|'),
    'credit': ('ACT_CD_CREDIT_MDM28032025.txt', '|'),
    'getchq': ('ACT_CD_GETCHQ_MDM28032025.txt', '|'),
    'operations': ('ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt', ';'),
    'vignette': ('ACT_CD_PAI_VIGNETTE_MDM20032025.txt', '|'),
    'produit': ('ACT_CD_PRODUIT_MDM20032025.txt', '|'),
    'solde': ('ACT_CD_SOLDE_CLIENT_MDM28032025.txt', '|'),
    'connexion': ('ACT_CD_STAT_CONNEXION_MDM28032025.txt', '|'),
    'transa': ('ACT_CD_TRANSA_MDM20032025.txt', '|'),
    'transferts': ('ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt', '|'),
    'equi': ('ACT_EQUI_MDM28032025.txt', '|')
}

# Encodages à tester
encodages = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16', 'cp850']

# Dictionnaire pour stocker les DataFrames
dfs = {}

# Fonction pour détecter automatiquement l'encodage
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

# Chargement des fichiers avec détection d'encodage
for nom, (fichier, separateur) in fichiers.items():
    file_path = os.path.join(data_path, fichier)
    print(f"Traitement du fichier {fichier}...")
    
    # Essai de détection automatique
    try:
        detected_encoding = detect_encoding(file_path)
        print(f"  Encodage détecté: {detected_encoding}")
        dfs[nom] = pd.read_csv(file_path, sep=separateur, encoding=detected_encoding)
        print(f"  Chargé avec succès en utilisant l'encodage détecté.")
        continue
    except Exception as e:
        print(f"  Échec avec l'encodage détecté: {e}")
    
    # Essai des encodages de la liste
    success = False
    for enc in encodages:
        try:
            print(f"  Essai avec l'encodage {enc}...")
            dfs[nom] = pd.read_csv(file_path, sep=separateur, encoding=enc)
            print(f"  Succès avec l'encodage {enc}!")
            success = True
            break
        except Exception as e:
            print(f"  Échec avec l'encodage {enc}: {e}")
    
    if not success:
        print(f"ERREUR: Impossible de charger le fichier {fichier} avec les encodages disponibles.")
        exit(1)

print("\nTous les fichiers ont été chargés avec succès!")

# Effectuer les LEFT JOINs
# Commencer avec le périmètre comme base
resultat = dfs['perimetre']

# JOIN avec la table crédit
resultat = pd.merge(resultat, dfs['credit'], on='RADICAL', how='left')

# JOIN avec la table getchq (CONTRAT = RADICAL)
resultat = pd.merge(resultat, dfs['getchq'].rename(columns={'CONTRAT': 'RADICAL'}), on='RADICAL', how='left')

# JOIN avec la table opérations
resultat = pd.merge(resultat, dfs['operations'], on='RADICAL', how='left')

# JOIN avec la table vignette
resultat = pd.merge(resultat, dfs['vignette'], on='RADICAL', how='left')

# JOIN avec la table produit
resultat = pd.merge(resultat, dfs['produit'], on='RADICAL', how='left')

# JOIN avec la table solde
resultat = pd.merge(resultat, dfs['solde'], on='RADICAL', how='left')

# JOIN avec la table connexion (CONTRAT = RADICAL)
resultat = pd.merge(resultat, dfs['connexion'].rename(columns={'CONTRAT': 'RADICAL'}), on='RADICAL', how='left')

# JOIN avec la table transa
resultat = pd.merge(resultat, dfs['transa'], on='RADICAL', how='left')

# JOIN avec la table transferts
resultat = pd.merge(resultat, dfs['transferts'], on='RADICAL', how='left')

# JOIN avec la table equi
resultat = pd.merge(resultat, dfs['equi'], on='RADICAL', how='left')

# Enregistrer le résultat final
try:
    resultat.to_csv(os.path.join(data_path, 'resultat_jointures_mdm.csv'), index=False)
    print(f"Résultat enregistré avec succès dans {os.path.join(data_path, 'resultat_jointures_mdm.csv')}")
except Exception as e:
    print(f"Erreur lors de l'enregistrement du résultat: {e}")
    # Essayer avec un encodage différent
    try:
        resultat.to_csv(os.path.join(data_path, 'resultat_jointures_mdm.csv'), index=False, encoding='latin-1')
        print(f"Résultat enregistré avec succès en utilisant l'encodage latin-1")
    except:
        print("Impossible d'enregistrer le résultat.")

print(f"Nombre de lignes dans le résultat final: {len(resultat)}")
print(f"Nombre de colonnes dans le résultat final: {len(resultat.columns)}")

Traitement du fichier Périmétre_projet.txt...
  Encodage détecté: utf-8
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_CREDIT_MDM28032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_GETCHQ_MDM28032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_PAI_VIGNETTE_MDM20032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_PRODUIT_MDM20032025.txt...
  Encodage détecté: ISO-8859-1


/tmp/ipykernel_18830/293873947.py:44: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[nom] = pd.read_csv(file_path, sep=separateur, encoding=detected_encoding)


  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_SOLDE_CLIENT_MDM28032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_STAT_CONNEXION_MDM28032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_TRANSA_MDM20032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.
Traitement du fichier ACT_EQUI_MDM28032025.txt...
  Encodage détecté: ascii
  Chargé avec succès en utilisant l'encodage détecté.

Tous les fichiers ont été chargés avec succès!


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import concurrent.futures
from functools import partial

def load_file(file_path, separator):
    """Charge un fichier avec gestion des séparateurs et encodages"""
    if not os.path.exists(file_path):
        print(f"Fichier introuvable: {file_path}")
        return pd.DataFrame()
        
    encodings = ['utf-8', 'latin1', 'ISO-8859-1']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, sep=separator, encoding=encoding, 
                            low_memory=False, on_bad_lines='skip')
            df.columns = df.columns.str.strip()
            print(f"Fichier chargé avec succès: {file_path}")
            
            # Optimisation: convertir directement les colonnes lors du chargement
            df = convert_columns(df)
            return df
        except Exception as e:
            print(f"Erreur lors du chargement avec l'encoding {encoding}: {str(e)}")
            continue
            
    print(f"Impossible de charger le fichier {file_path} avec les encodages testés")
    return pd.DataFrame()

def convert_columns(df):
    """Convertit les colonnes numériques et dates en une seule passe"""
    # Colonnes numériques potentielles
    numeric_cols = ['MONTANT', 'NOMBRE', 'MONTANT_TOTAL', 'SOLDE', 'MVTDEB', 'MVTCRED', 'NOMBRE_OPE', 'NOMBRE_OPERATION', 'CONNECTIONCOUNT']
    
    # Colonnes de dates potentielles
    date_cols = ['DATE_OF_BIRTH', 'CUSTOMER_SINCE', 'DT_CONFIRMATION', 
                'DATE_OCTROI', 'DT_FIN_DOSS_REEL', 'DATE_DEM_ABON', 
                'DATE_RES_ABON', 'DATE_VAL_ABON', 'CREATEDATE', 'LASTLOGINDATE', 'DT_CREATE', 'DT_ENR']
    
    date_formats = ['%Y%m%d', '%d/%m/%Y', '%m/%y', '%d/%m/%y', '%Y-%m-%d']
    
    # Correction encodage pour colonnes textuelles
    for col in df.columns:
        if df[col].dtype == 'object':
            # Convertir les colonnes numériques
            if col in numeric_cols:
                df[col] = pd.to_numeric(df[col].astype(str)
                                        .str.replace(',', '.')
                                        .str.replace(' ', ''), 
                                      errors='coerce')
                                      
            # Convertir les colonnes de date
            elif col in date_cols:
                for date_format in date_formats:
                    try:
                        df[col] = pd.to_datetime(df[col], format=date_format, errors='coerce')
                        break
                    except:
                        continue
                        
            # Corriger l'encodage des chaînes de caractères restantes
            else:
                try:
                    df[col] = df[col].apply(lambda x: x.encode('latin1', errors='ignore')
                                           .decode('utf-8', errors='ignore') 
                                           if isinstance(x, str) else x)
                except:
                    pass
    
    return df

def preserve_identifiers_format(df, id_columns):
    """Préserve le format des identifiants en assurant qu'ils ont 7 chiffres avec des zéros au début si nécessaire"""
    for col in id_columns:
        if col in df.columns:
            # Conversion des identifiants en chaînes de caractères
            df[col] = df[col].astype(str)
            
            # Supprimer les espaces et NaN
            df[col] = df[col].str.strip()
            df[col] = df[col].replace('nan', '')
            
            # Appliquer le padding pour assurer 7 chiffres
            df[col] = df[col].apply(lambda x: x.zfill(7) if x and x.isdigit() else x)
    
    return df

def load_all_files_parallel(data_files):
    """Charge tous les fichiers en parallèle"""
    os.makedirs('data', exist_ok=True)
    
    df_dict = {}
    missing_files = []
    
    # Fonction partielle pour le multiprocessing
    load_file_wrapper = lambda args: (args[0], load_file(args[1], args[2]))
    
    # Charger les fichiers en parallèle
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(data_files), os.cpu_count() or 4)) as executor:
        results = executor.map(load_file_wrapper, data_files)
        
    for name, df in results:
        if not df.empty:
            # Préserver le format des identifiants
            id_columns = ['RADICAL', 'CONTRAT']
            df = preserve_identifiers_format(df, id_columns)
            df_dict[name] = df
        else:
            missing_files.append(name)
    
    return df_dict, missing_files

def main():
    # Définition des fichiers à traiter
    data_files = [
        ('transferts', 'data/ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt', '|'),
        ('connexions', 'data/ACT_CD_STAT_CONNEXION_MDM28032025.txt', '|'),
        ('perimetre', 'data/Périmétre_projet.txt', '|'),
        ('soldes', 'data/ACT_CD_SOLDE_CLIENT_MDM28032025.txt', '|'),
        ('equipements', 'data/ACT_EQUI_MDM28032025.txt', '|'),
        ('transactions', 'data/ACT_CD_TRANSA_MDM20032025.txt', '|'),
        ('produits', 'data/ACT_CD_PRODUIT_MDM20032025.txt', '|'),
        ('operations', 'data/ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt', ';'),
        ('vignettes', 'data/ACT_CD_PAI_VIGNETTE_MDM20032025.txt', '|'),
        ('cheques', 'data/ACT_CD_GETCHQ_MDM28032025.txt', '|'),
        ('credits', 'data/ACT_CD_CREDIT_MDM28032025.txt', '|')
    ]
    
    try:
        # Chargement parallèle des fichiers
        print("Chargement des fichiers en parallèle...")
        df_dict, missing_files = load_all_files_parallel(data_files)
        
        # Vérifier les fichiers critiques
        if missing_files:
            print("\nFichiers non chargés:")
            for file_path in missing_files:
                print(f"- {file_path}")
            
            critical_files = ['perimetre', 'connexions', 'transactions', 'cheques', 'vignettes', 'equipements']
            missing_critical = [file for file in critical_files if file not in df_dict]
            if missing_critical:
                print(f"\nImpossible de continuer: fichiers nécessaires manquants: {', '.join(missing_critical)}")
                return
        
        # IMPORTANT: S'assurer que le fichier de périmètre est présent
        if 'perimetre' not in df_dict:
            print("Fichier périmètre manquant - impossible de continuer")
            return
        
        # Afficher un aperçu du fichier équipements pour debug
        if 'equipements' in df_dict:
            print("\nAperçu du fichier équipements:")
            print(f"Colonnes disponibles: {df_dict['equipements'].columns.tolist()}")
            
            # Afficher la distribution des valeurs de ETAT et LIBELLE
            if 'ETAT' in df_dict['equipements'].columns and 'LIBELLE' in df_dict['equipements'].columns:
                print("\nDistribution des valeurs dans ETAT:")
                etat_status = df_dict['equipements']['ETAT'].value_counts()
                print(etat_status)
                
                print("\nDistribution des valeurs dans LIBELLE:")
                # Afficher quelques valeurs brutes pour vérifier les espaces
                print("Exemples de valeurs dans LIBELLE (avec représentation des caractères invisibles):")
                libelle_samples = df_dict['equipements']['LIBELLE'].head(10).apply(lambda x: repr(x) if isinstance(x, str) else x)
                print(libelle_samples)
                
                libelle_status = df_dict['equipements']['LIBELLE'].value_counts().head(10)  # Afficher les 10 plus fréquents
                print(libelle_status)
                
                # Nombre total d'équipements validés
                # MODIFICATION: Utiliser la méthode str.strip() et str.upper() pour normaliser les valeurs
                validations = df_dict['equipements'][
                    (df_dict['equipements']['ETAT'].astype(str).str.strip() == '2') & 
                    (df_dict['equipements']['LIBELLE'].astype(str).str.strip().str.upper() == 'VALIDE')
                ].shape[0]
                print(f"Nombre total d'équipements validés (ETAT=2, LIBELLE=VALIDE): {validations}")
        
        # Vérifier le format des identifiants
        print("\nVérification du format des identifiants...")
        for name, df in df_dict.items():
            if 'RADICAL' in df.columns:
                sample_radical = df['RADICAL'].iloc[0] if not df.empty else "N/A"
                print(f"- {name}: Format RADICAL exemple: {sample_radical}")
                
            if 'CONTRAT' in df.columns:
                sample_contrat = df['CONTRAT'].iloc[0] if not df.empty else "N/A"
                print(f"- {name}: Format CONTRAT exemple: {sample_contrat}")
        
        # MODIFICATION: Extraire clairement tous les identifiants du périmètre
        print("\nIdentification des clients du périmètre...")
        contrat_col = 'CONTRAT'
        
        # Récupérer tous les identifiants du fichier périmètre
        perimetre_df = df_dict['perimetre']
        
        # Assurer que les identifiants sont du bon format (7 chiffres)
        perimetre_df = preserve_identifiers_format(perimetre_df, ['RADICAL', contrat_col])
        
        # Collecter tous les identifiants du périmètre (RADICAL et CONTRAT)
        perimetre_identifiers = set(perimetre_df['RADICAL'].dropna().unique())
        
        # Ajouter les contrats associés s'ils existent
        if contrat_col in perimetre_df.columns:
            perimetre_identifiers.update(perimetre_df[contrat_col].dropna().unique())
        
        print(f"Nombre de clients dans le périmètre: {len(perimetre_identifiers)}")
        
        # Optimisation: créer une table de correspondance unique pour les identifiants
        # Dictionnaire pour stocker les correspondances RADICAL-CONTRAT
        identifier_map = {}
        
        # Construire la table de correspondance en une seule passe
        for name, df in df_dict.items():
            if 'RADICAL' in df.columns and contrat_col in df.columns:
                print(f"Extraction des relations RADICAL-CONTRAT depuis {name}...")
                
                # Optimisation: utiliser des opérations vectorisées
                matched_rows = df[df['RADICAL'].notna() & df[contrat_col].notna()]
                
                for _, row in matched_rows.iterrows():
                    radical = row['RADICAL']
                    contrat = row[contrat_col]
                    
                    # Enregistrer uniquement les correspondances pour les clients du périmètre
                    if radical in perimetre_identifiers or contrat in perimetre_identifiers:
                        # Stocker les correspondances dans les deux sens
                        identifier_map[radical] = contrat
                        identifier_map[contrat] = radical
        
        # 1. Clients avec transactions
        print("\nIdentification des clients avec transactions...")
        transaction_clients = set()
        if 'transactions' in df_dict:
            transactions_df = df_dict['transactions']
            if 'RADICAL' in transactions_df.columns and 'NOMBRE_OPE' in transactions_df.columns:
                # Filtrer les clients avec au moins une opération
                transaction_mask = (transactions_df['RADICAL'].notna() & 
                                   (transactions_df['NOMBRE_OPE'] >= 1))
                
                transaction_identifiers = transactions_df.loc[transaction_mask, 'RADICAL']
                
                # Filtrer pour ne garder que les clients du périmètre
                transaction_clients.update(id for id in transaction_identifiers if id in perimetre_identifiers)
                
                # Ajouter les identifiants équivalents du périmètre
                for identifier in list(transaction_clients):
                    if identifier in identifier_map and identifier_map[identifier] in perimetre_identifiers:
                        transaction_clients.add(identifier_map[identifier])
        
        print(f"Nombre de clients avec transactions: {len(transaction_clients)}")
        
        # 2. Clients GETCHQ
        print("\nIdentification des clients GETCHQ...")
        getchq_clients = set()
        if 'cheques' in df_dict:
            cheques_df = df_dict['cheques']
            
            # Traiter les identifiants RADICAL
            if 'RADICAL' in cheques_df.columns:
                getchq_radical = cheques_df[cheques_df['RADICAL'].notna()]['RADICAL']
                getchq_clients.update(id for id in getchq_radical if id in perimetre_identifiers)
            
            # Traiter les identifiants CONTRAT
            if contrat_col in cheques_df.columns:
                getchq_contrat = cheques_df[cheques_df[contrat_col].notna()][contrat_col]
                getchq_clients.update(id for id in getchq_contrat if id in perimetre_identifiers)
            
            # Ajouter les identifiants équivalents du périmètre
            for identifier in list(getchq_clients):
                if identifier in identifier_map and identifier_map[identifier] in perimetre_identifiers:
                    getchq_clients.add(identifier_map[identifier])
        
        print(f"Nombre de clients GETCHQ: {len(getchq_clients)}")
        
        # 3. Clients vignettes
        print("\nIdentification des clients vignette...")
        vignette_clients = set()
        if 'vignettes' in df_dict:
            vignettes_df = df_dict['vignettes']
            if 'RADICAL' in vignettes_df.columns:
                vignette_radical = vignettes_df[vignettes_df['RADICAL'].notna()]['RADICAL']
                vignette_clients.update(id for id in vignette_radical if id in perimetre_identifiers)
                
                # Ajouter les identifiants équivalents du périmètre
                for identifier in list(vignette_clients):
                    if identifier in identifier_map and identifier_map[identifier] in perimetre_identifiers:
                        vignette_clients.add(identifier_map[identifier])
        
        print(f"Nombre de clients vignette: {len(vignette_clients)}")
        
        # 4. Clients avec abonnements validés
        print("\nIdentification des clients avec abonnements validés...")
        abonnement_clients = set()
        if 'equipements' in df_dict:
            equipements_df = df_dict['equipements']
            if 'RADICAL' in equipements_df.columns and 'ETAT' in equipements_df.columns and 'LIBELLE' in equipements_df.columns:
                # Nettoyer et normaliser les valeurs ETAT et LIBELLE
                equipements_df['ETAT_CLEAN'] = equipements_df['ETAT'].astype(str).str.strip()
                equipements_df['LIBELLE_CLEAN'] = equipements_df['LIBELLE'].astype(str).str.strip().str.upper()
                
                # Debug: afficher les valeurs distinctes après nettoyage
                print("\nValeurs distinctes dans ETAT après nettoyage:")
                print(equipements_df['ETAT_CLEAN'].value_counts())
                
                print("\nValeurs distinctes dans LIBELLE après nettoyage:")
                print(equipements_df['LIBELLE_CLEAN'].value_counts().head(10))
                
                # Filtrer pour les abonnements validés avec valeurs normalisées
                abonnement_mask = (equipements_df['RADICAL'].notna() & 
                                 (equipements_df['ETAT_CLEAN'] == '2') & 
                                 (equipements_df['LIBELLE_CLEAN'] == 'VALIDE'))
                
                # Debug: afficher le nombre de lignes qui correspondent à ces critères
                matching_rows = equipements_df[abonnement_mask]
                print(f"\nNombre de lignes avec ETAT=2 et LIBELLE=VALIDE: {len(matching_rows)}")
                
                if not matching_rows.empty:
                    print("Exemples de lignes correspondantes:")
                    print(matching_rows[['RADICAL', 'ETAT', 'LIBELLE', 'ETAT_CLEAN', 'LIBELLE_CLEAN']].head())
                
                abonnement_identifiers = equipements_df.loc[abonnement_mask, 'RADICAL']
                
                # Filtrer pour ne garder que les clients du périmètre
                abonnement_clients.update(id for id in abonnement_identifiers if id in perimetre_identifiers)
                
                # Debug: afficher quelques identifiants
                if abonnement_clients:
                    print("\nExemples d'identifiants de clients avec abonnements validés:")
                    print(list(abonnement_clients)[:5])
                
                # Ajouter les identifiants équivalents du périmètre
                for identifier in list(abonnement_clients):
                    if identifier in identifier_map and identifier_map[identifier] in perimetre_identifiers:
                        abonnement_clients.add(identifier_map[identifier])
        
        print(f"Nombre de clients avec abonnements validés: {len(abonnement_clients)}")
        
        # Si aucun abonnement trouvé, essayer une approche plus permissive
        if len(abonnement_clients) == 0 and 'equipements' in df_dict:
            print("\nAucun abonnement validé trouvé avec les critères exacts. Essai avec critères plus permissifs...")
            
            # Si le problème est qu'il n'y a pas de ligne exactement "VALIDE", prenons toutes les lignes avec ETAT=2
            equipements_df = df_dict['equipements']
            if 'RADICAL' in equipements_df.columns and 'ETAT' in equipements_df.columns:
                permissive_mask = (equipements_df['RADICAL'].notna() & 
                                 (equipements_df['ETAT'].astype(str).str.strip() == '2'))
                
                permissive_ids = equipements_df.loc[permissive_mask, 'RADICAL']
                
                # Filtrer pour ne garder que les clients du périmètre
                abonnement_clients.update(id for id in permissive_ids if id in perimetre_identifiers)
                
                # Ajouter les identifiants équivalents du périmètre
                for identifier in list(abonnement_clients):
                    if identifier in identifier_map and identifier_map[identifier] in perimetre_identifiers:
                        abonnement_clients.add(identifier_map[identifier])
                
                print(f"Nombre de clients avec abonnements (critère permissif ETAT=2): {len(abonnement_clients)}")
        
        # MODIFICATION: Traitement des clients connectés séparément (information uniquement, pas utilisée pour l'appétence)
        print("\nIdentification des clients connectés (information uniquement)...")
        connected_clients = set()
        if 'connexions' in df_dict:
            connexions_df = df_dict['connexions']
            if contrat_col in connexions_df.columns:
                # Un client est considéré comme connecté si:
                # 1. Il a une valeur 'Connecte Reussi' dans CONNECTION, OU
                # 2. Il a un nombre de connexions CONNECTIONCOUNT > 0
                
                # Condition 1: CONNECTION contient 'Connecte Reussi'
                if 'CONNECTION' in connexions_df.columns:
                    connected_mask1 = connexions_df[contrat_col].notna() & connexions_df['CONNECTION'].str.contains('Connecte Reussi', case=False, na=False)
                    connected_ids1 = set(connexions_df.loc[connected_mask1, contrat_col])
                    print(f"Nombre de clients avec 'Connecte Reussi': {len(connected_ids1)}")
                    connected_clients.update(id for id in connected_ids1 if id in perimetre_identifiers)
                
                # Condition 2: CONNECTIONCOUNT > 0
                if 'CONNECTIONCOUNT' in connexions_df.columns:
                    connexions_df['CONNECTIONCOUNT'] = pd.to_numeric(connexions_df['CONNECTIONCOUNT'], errors='coerce')
                    connected_mask2 = connexions_df[contrat_col].notna() & (connexions_df['CONNECTIONCOUNT'] > 0)
                    connected_ids2 = set(connexions_df.loc[connected_mask2, contrat_col])
                    print(f"Nombre de clients avec CONNECTIONCOUNT > 0: {len(connected_ids2)}")
                    connected_clients.update(id for id in connected_ids2 if id in perimetre_identifiers)
                
                # Ou utiliser les deux colonnes CREATEDATE et LASTLOGINDATE si elles existent et sont différentes
                if 'CREATEDATE' in connexions_df.columns and 'LASTLOGINDATE' in connexions_df.columns:
                    # Un client qui s'est connecté aura des dates différentes
                    date_mask = (connexions_df[contrat_col].notna() & 
                                  connexions_df['CREATEDATE'].notna() & 
                                  connexions_df['LASTLOGINDATE'].notna() & 
                                  (connexions_df['CREATEDATE'] != connexions_df['LASTLOGINDATE']))
                    connected_ids3 = set(connexions_df.loc[date_mask, contrat_col])
                    print(f"Nombre de clients avec dates différentes: {len(connected_ids3)}")
                    connected_clients.update(id for id in connected_ids3 if id in perimetre_identifiers)
                
                # Ajouter les identifiants équivalents du périmètre
                for identifier in list(connected_clients):
                    if identifier in identifier_map and identifier_map[identifier] in perimetre_identifiers:
                        connected_clients.add(identifier_map[identifier])
        
        print(f"Nombre total de clients connectés: {len(connected_clients)}")
        
        # Construction du dataset unifié
        print("\nConstruction du dataset unifié...")
        
        # Préparation du DataFrame final
        unified_data = {
            'IDENTIFIER': list(perimetre_identifiers),  # Tous les clients du périmètre
            'connected': [id in connected_clients for id in perimetre_identifiers],  # Info connexion (pas utilisée pour appétence)
            'has_transaction': [id in transaction_clients for id in perimetre_identifiers],
            'used_getchq': [id in getchq_clients for id in perimetre_identifiers],
            'paid_vignette': [id in vignette_clients for id in perimetre_identifiers],
            'validated_subscription': [id in abonnement_clients for id in perimetre_identifiers],
        }
        
        # MODIFICATION: Calculer l'appétence basée sur la nouvelle définition
# Un client est appétent s'il:
# 1. Est dans le périmètre (déjà filtré dans unified_data)
# 2. A un abonnement validé ET
# 3. A réalisé au moins une des actions suivantes:
#    - Au moins une transaction
#    - Utilisé le service GETCHQ
#    - Payé une vignette en ligne
        unified_data['has_any_action'] = [(t or g or v) for t, g, v in 
                               zip(unified_data['has_transaction'],
                                   unified_data['used_getchq'],
                                   unified_data['paid_vignette'])]

        unified_data['appetent'] = [(s and a) for s, a in 
                          zip(unified_data['validated_subscription'],
                              unified_data['has_any_action'])]
        
        # Créer le DataFrame unifié
        unified_df = pd.DataFrame(unified_data)
        
        # Joindre les données de périmètre
        perimetre_df = df_dict['perimetre'].copy()
        
        # Créer une version "longue" de perimetre avec RADICAL et CONTRAT séparément
        radical_perim = perimetre_df[['RADICAL', 'GENRE', 'STATUT_MARITAL', 'DATE_OF_BIRTH', 
                                     'CTRY_OF_BIRTH', 'RESIDENCE', 'CR_PROFIL', 'CUSTOMER_SINCE']]
        radical_perim.rename(columns={'RADICAL': 'IDENTIFIER'}, inplace=True)
        
        if contrat_col in perimetre_df.columns:
            contrat_perim = perimetre_df[[contrat_col, 'GENRE', 'STATUT_MARITAL', 'DATE_OF_BIRTH', 
                                        'CTRY_OF_BIRTH', 'RESIDENCE', 'CR_PROFIL', 'CUSTOMER_SINCE']]
            contrat_perim.rename(columns={contrat_col: 'IDENTIFIER'}, inplace=True)
            
            # Concaténer les deux versions
            perim_long = pd.concat([radical_perim, contrat_perim], ignore_index=True)
        else:
            perim_long = radical_perim
        
        # Supprimer les doublons
        perim_long.drop_duplicates(subset=['IDENTIFIER'], inplace=True)
        
        # Jointure gauche pour conserver tous les clients du périmètre
        unified_df = pd.merge(unified_df, perim_long, on='IDENTIFIER', how='left')
        
        # Sauvegarder le dataset unifié
        output_file = 'dataset_unifie_perimetre_appetent222.csv'
        unified_df.to_csv(output_file, index=False)
        print(f"\nDataset unifié sauvegardé dans {output_file}")
        
        # Afficher les statistiques
        appetent_count = unified_df['appetent'].sum()
        total_count = len(unified_df)
        print(f"\nNombre total de clients du périmètre: {total_count}")
        print(f"Nombre de clients appétents: {appetent_count} ({appetent_count/total_count*100:.2f}%)")
        print(f"Nombre de clients non appétents: {total_count - appetent_count} ({(total_count - appetent_count)/total_count*100:.2f}%)")
        
        # Afficher la distribution des profils si disponible
        if 'CR_PROFIL' in unified_df.columns:
            print("\nDistribution des profils:")
            print(unified_df['CR_PROFIL'].value_counts())
        
        # Afficher statistiques des facteurs d'appétence
        print("\nStatistiques des facteurs d'appétence:")
        print(f"- Clients connectés: {unified_df['connected'].sum()} ({unified_df['connected'].sum()/total_count*100:.2f}%) [Info uniquement]")
        print(f"- Clients avec transactions: {unified_df['has_transaction'].sum()} ({unified_df['has_transaction'].sum()/total_count*100:.2f}%)")
        print(f"- Clients GETCHQ: {unified_df['used_getchq'].sum()} ({unified_df['used_getchq'].sum()/total_count*100:.2f}%)")
        print(f"- Clients vignette: {unified_df['paid_vignette'].sum()} ({unified_df['paid_vignette'].sum()/total_count*100:.2f}%)")
        print(f"- Clients avec abonnements validés: {unified_df['validated_subscription'].sum()} ({unified_df['validated_subscription'].sum()/total_count*100:.2f}%)")
        
        
        # Analyse des combinaisons de critères
        print("\nAnalyse des combinaisons de critères d'appétence:")
        only_transactions = unified_df[unified_df['has_transaction'] & ~(unified_df['used_getchq'] | unified_df['paid_vignette'] | unified_df['validated_subscription'])]
        print(f"- Clients avec uniquement des transactions: {len(only_transactions)} ({len(only_transactions)/total_count*100:.2f}%)")
        
        only_getchq = unified_df[unified_df['used_getchq'] & ~(unified_df['has_transaction'] | unified_df['paid_vignette'] | unified_df['validated_subscription'])]
        print(f"- Clients ayant uniquement utilisé GETCHQ: {len(only_getchq)} ({len(only_getchq)/total_count*100:.2f}%)")
        
        only_vignette = unified_df[unified_df['paid_vignette'] & ~(unified_df['has_transaction'] | unified_df['used_getchq'] | unified_df['validated_subscription'])]
        print(f"- Clients ayant uniquement payé une vignette: {len(only_vignette)} ({len(only_vignette)/total_count*100:.2f}%)")
        
        only_subscription = unified_df[unified_df['validated_subscription'] & ~(unified_df['has_transaction'] | unified_df['used_getchq'] | unified_df['paid_vignette'])]
        print(f"- Clients ayant uniquement des abonnements validés: {len(only_subscription)} ({len(only_subscription)/total_count*100:.2f}%)")
        
        all_actions = unified_df[unified_df['has_transaction'] & unified_df['used_getchq'] & unified_df['paid_vignette'] & unified_df['validated_subscription']]
        print(f"- Clients avec les 4 actions: {len(all_actions)} ({len(all_actions)/total_count*100:.2f}%)")
        
        # Analyse des clients connectés vs. appétents
        connected_appetent = unified_df[unified_df['connected'] & unified_df['appetent']]
        print(f"- Clients connectés ET appétents: {len(connected_appetent)} ({len(connected_appetent)/total_count*100:.2f}%)")
        
        not_connected_appetent = unified_df[~unified_df['connected'] & unified_df['appetent']]
        print(f"- Clients non connectés mais appétents: {len(not_connected_appetent)} ({len(not_connected_appetent)/total_count*100:.2f}%)")
        
        # Analyse par profil
        if 'CR_PROFIL' in unified_df.columns:
            print("\nTaux d'appétence par profil client:")
            appetence_par_profil = unified_df.groupby('CR_PROFIL')['appetent'].mean() * 100
            print(appetence_par_profil)
    
    except Exception as e:
        import traceback
        print(f"\nErreur lors du traitement: {e}")
        print("\nDétails de l'erreur:")
        traceback.print_exc()

if _name_ == "_main_":
    main()

NameError: name '_name_' is not defined